In [1]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
#import rasterio
#from rasterio.mask import mask

In [2]:
project_root = os.path.dirname(os.path.dirname("test.ipynb"))
data_dir = os.path.join(project_root, "data")
raster_dir = os.path.join(data_dir, "rasters")
transient_dir = os.path.join(project_root, "transients")
output_dir = os.path.join(project_root, "outputs")

energy_raster_path = os.path.join(raster_dir, "GHS_BUILT_S_timeseries_points.gpkg")


shapefile_path = os.path.join(data_dir, "ne_10m_admin_0_countries.shp")  # may need other files rather than just shp?
raster_path = os.path.join(data_dir, "gpw_v4_population_density_rev11_2020_30_min.tif")
country_energy_path = os.path.join(data_dir, "Country Energy Data.xlsx")

In [3]:
energy_timeseries = gpd.read_file(energy_raster_path)

led_data = pd.read_excel(country_energy_path)
chosen_column = [str(col) for col in led_data.columns if "Chosen" in str(col)][0]
led_data = led_data[led_data[chosen_column] > 0].dropna(subset=[chosen_column])


year = 2025
place_ocean = True
all_leds_gdf = gpd.GeoDataFrame()

c:\ProgramData\miniforge3\envs\science_gen\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [4]:
energy_timeseries

,point_index,country,1975,1980,1985,1990,1995,2000,2005,2010,2015,2020,2025,geometry
0,0,,0,0,0,0,0,0,0,0,0,0,0,POINT (-179.75125 88.84958)
1,1,None,0,0,0,0,0,0,0,0,0,0,0,POINT (-179.25125 88.84958)
2,2,None,0,0,0,0,0,0,0,0,0,0,0,POINT (-178.75125 88.84958)
3,3,None,0,0,0,0,0,0,0,0,0,0,0,POINT (-178.25125 88.84958)
4,4,None,0,0,0,0,0,0,0,0,0,0,0,POINT (-177.75125 88.84958)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256316,256315,Antarctica,0,0,0,0,0,0,0,0,0,0,0,POINT (177.74875 -88.65042)
256317,256316,Antarctica,0,0,0,0,0,0,0,0,0,0,0,POINT (178.24875 -88.65042)
256318,256317,Antarctica,0,0,0,0,0,0,0,0,0,0,0,POINT (178.74875 -88.65042)
256319,256318,Antarctica,0,0,0,0,0,0,0,0,0,0,0,POINT (179.24875 -88.65042)


In [5]:
for index, row in led_data.iterrows():

    country_name = row['Entity']
    num_leds = int(row['Round'])
    leds_placed = 0

    values_array = energy_timeseries[energy_timeseries['country'] == country_name]
    values_array = values_array[["point_index", f"{year}", "geometry"]].sort_values(f"{year}", ascending=False)

    available_cells = len(values_array)
    missing_leds = num_leds - available_cells
    
    if available_cells == 0:
        print(f"Could not find {country_name} in raster data, skipping...")

    else:

        for leds in range(0,num_leds-leds_placed): # Place LEDs on the land-space

            if leds_placed < available_cells: 
                
                all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_array["geometry"].iloc[leds]],
                                                                        'Country': [country_name],
                                                                        'Raster_Density': [values_array[f"{year}"].iloc[leds]]
                                                                        }, geometry='geometry')], ignore_index=True)
                leds_placed += 1

            else:

                if place_ocean == True:
                    
                    if leds_placed >= num_leds:
                        break

                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {available_cells} cells are available. Attempting to place {missing_leds} LEDs in surrounding area.")
                    values_sorted = energy_timeseries.iloc[energy_timeseries.geometry.x.argsort().values].reset_index(drop=True)
                    surround = 1
                    
                    while leds_placed < num_leds:

                        values_sorted_filtered = values_sorted[values_sorted['country'] == country_name]
                        surround_indices = [x-surround for x in values_sorted_filtered.index] + [x+surround for x in values_sorted_filtered.index] 
                        surround_indices = values_sorted.loc[surround_indices].query("country.isnull()").index
                        values_sorted.loc[surround_indices, "country"] = country_name

                        for leds in surround_indices: # Place remaining LEDs on the land-space
                            all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_sorted["geometry"].iloc[leds]],
                                                                                    'Country': [country_name],
                                                                                    'Raster_Density': [values_sorted[f"{year}"].iloc[leds]]
                                                                                    }, geometry='geometry')], ignore_index=True)
                            leds_placed += 1
                            if leds_placed >= num_leds:
                                break
                        surround += 1
                
                else: 
                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {leds_placed} were placed due to not having enough space.")
                    break

Could not find Hong Kong S.A.R. in raster data, skipping...
Could not find Bahrain in raster data, skipping...
Could not find Republic of Serbia in raster data, skipping...
Could not find Dominican Republic in raster data, skipping...
Could not find Bosnia and Herzegovina in raster data, skipping...
Could not find Ivory Coast in raster data, skipping...
Could not find United Republic of Tanzania in raster data, skipping...
Could not find Netherlands Antilles in raster data, skipping...


In [6]:
all_leds_gdf.to_file("dataframe.gpkg", driver="GPKG")

c:\ProgramData\miniforge3\envs\science_gen\Lib\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
